# Pull CRSP Monthly Returns + Compustat Controls for 10-K Sample

This notebook:
1. Reads filing dates from `{cik}_filings.parquet` files
2. Reads S&P 500 membership from `usable_pairs.parquet`
3. Maps CIK → PERMNO using WRDS
4. Pulls CRSP monthly returns for each firm for the **12 months after** each 10-K filing date
5. Pulls Compustat accounting data (at, ni, sale) for controls
6. Saves a clean panel: `crsp_returns_panel.parquet`

**Requires**: WRDS account with access to CRSP and Compustat.

## 0. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


## 1. Configuration

In [2]:
import os

BASE = '/content/drive/MyDrive/FML_project_4'

CONFIG = {
    'filings_folder':    BASE,
    'usable_pairs_path': os.path.join(BASE, 'usable_pairs.parquet'),
    'output_path':       os.path.join(BASE, 'crsp_returns_panel.parquet'),
    'filing_dates_path': os.path.join(BASE, 'filing_dates.parquet'),
    'start_year': 2004,
    'end_year':   2025,
}
print('Config OK')

Config OK


## 2. Dependencies

In [3]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'wrds', 'pyarrow', 'tqdm'], check=True)

import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
print('OK')

OK


## 3. Extract filing dates from parquet files

Each `{cik}_filings.parquet` has columns including `cik`, `year`, `filing_date`.
We extract one row per (cik, year) — the actual SEC filing date.

In [4]:
pq_files = sorted(glob.glob(
    os.path.join(CONFIG['filings_folder'], '**', '*_filings.parquet'), recursive=True
))
print(f'Filing parquets found: {len(pq_files)}')

filing_rows = []
for f in tqdm(pq_files, desc='Reading filing dates'):
    try:
        df = pd.read_parquet(f, columns=['cik', 'year', 'filing_date'])
        filing_rows.append(df)
    except Exception:
        continue

filings = pd.concat(filing_rows, ignore_index=True)
filings['cik']         = filings['cik'].astype(str).str.strip().str.lstrip('0')
filings['year']        = pd.to_numeric(filings['year'], errors='coerce').astype('Int64')
filings['filing_date'] = pd.to_datetime(filings['filing_date'], errors='coerce')
filings = filings.dropna(subset=['filing_date'])
filings = filings[filings['year'].between(CONFIG['start_year'], CONFIG['end_year'])]

# Keep latest filing per (cik, year) in case of amendments
filings = (filings.sort_values('filing_date')
                   .drop_duplicates(subset=['cik', 'year'], keep='last')
                   .reset_index(drop=True))

print(f'Filing dates: {len(filings):,} filings  |  {filings["cik"].nunique():,} CIKs')
print(f'Date range: {filings["filing_date"].min().date()} – {filings["filing_date"].max().date()}')

# Apply S&P 500 membership filter
usable = pd.read_parquet(CONFIG['usable_pairs_path'])
usable['cik']  = usable['cik'].astype(str).str.strip().str.lstrip('0')
usable['year'] = usable['year'].astype('Int64')
usable_set = set(zip(usable['cik'], usable['year'].astype(int)))

filings = filings[
    filings.apply(lambda r: (r['cik'], int(r['year'])) in usable_set, axis=1)
].reset_index(drop=True)

print(f'After S&P 500 filter: {len(filings):,} filings  |  {filings["cik"].nunique():,} CIKs')

# Save for reuse
filings.to_parquet(CONFIG['filing_dates_path'], index=False)
print(f'Saved: {CONFIG["filing_dates_path"]}')

Filing parquets found: 977


Reading filing dates:   0%|          | 0/977 [00:00<?, ?it/s]

Filing dates: 15,556 filings  |  976 CIKs
Date range: 2004-01-15 – 2025-12-22
After S&P 500 filter: 10,264 filings  |  897 CIKs
Saved: /content/drive/MyDrive/FML_project_4/filing_dates.parquet


## 4. Connect to WRDS and map CIK → PERMNO

We need PERMNO to pull CRSP returns. The SEC-CRSP link table maps CIK to PERMNO.

In [9]:
import wrds

db = wrds.Connection()
print('Connected to WRDS')

# ── CIK to PERMNO mapping via WRDS SEC-CRSP link ──────────────────
unique_ciks = filings['cik'].unique().tolist()
print(f'Unique CIKs to map: {len(unique_ciks)}')

# Try the SEC-CRSP linking table
comp = db.raw_sql("""
    SELECT gvkey, cik
    FROM comp.names
    WHERE cik IS NOT NULL
""")

link = db.raw_sql("""
    SELECT gvkey, lpermno AS permno
    FROM crsp.ccmxpf_linktable
    WHERE lpermno IS NOT NULL
""")

merged = comp.merge(link, on='gvkey')

merged['cik'] = merged['cik'].astype(str).str.strip().str.lstrip('0')
merged['permno'] = merged['permno'].astype(int)
merged = merged.drop_duplicates(subset=['cik'], keep='first')

filings = filings.merge(merged, on='cik', how='left')
n_mapped = filings['permno'].notna().sum()
print(f'Mapped CIK→PERMNO: {n_mapped:,} / {len(filings):,} filings')
print(f'Unmapped CIKs: {filings[filings["permno"].isna()]["cik"].nunique()}')

# Drop unmapped
filings = filings.dropna(subset=['permno'])
filings['permno'] = filings['permno'].astype(int)
print(f'Final: {len(filings):,} filings  |  {filings["cik"].nunique()} CIKs  |  '
      f'{filings["permno"].nunique()} PERMNOs')

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Connected to WRDS
Unique CIKs to map: 897
Mapped CIK→PERMNO: 10,264 / 10,264 filings
Unmapped CIKs: 0
Final: 10,264 filings  |  897 CIKs  |  890 PERMNOs


## 5. Pull CRSP monthly returns

For each filing, we need the 12 monthly returns **after** the filing date.
We pull the full CRSP monthly panel for our PERMNOs and date range,
then align in step 7.

In [10]:
permnos = filings['permno'].unique().tolist()
min_date = filings['filing_date'].min() - pd.DateOffset(months=13)  # for momentum
max_date = filings['filing_date'].max() + pd.DateOffset(months=13)

print(f'Pulling CRSP for {len(permnos)} PERMNOs')
print(f'Date range: {min_date.date()} – {max_date.date()}')

# Pull in chunks to avoid SQL size limits
chunk_size = 500
crsp_parts = []

for i in tqdm(range(0, len(permnos), chunk_size), desc='CRSP chunks'):
    chunk = permnos[i:i+chunk_size]
    permno_str = ','.join(str(p) for p in chunk)
    query = f"""
        SELECT permno, date, ret, prc, shrout
        FROM crsp.msf
        WHERE permno IN ({permno_str})
          AND date BETWEEN '{min_date.strftime('%Y-%m-%d')}'
                       AND '{max_date.strftime('%Y-%m-%d')}'
    """
    part = db.raw_sql(query)
    crsp_parts.append(part)

crsp = pd.concat(crsp_parts, ignore_index=True)
crsp['date']   = pd.to_datetime(crsp['date'])
crsp['permno'] = crsp['permno'].astype(int)
crsp['ret']    = pd.to_numeric(crsp['ret'], errors='coerce')
# Market cap in $millions
crsp['mktcap'] = (crsp['prc'].abs() * crsp['shrout']) / 1000

print(f'CRSP monthly: {len(crsp):,} rows  |  {crsp["permno"].nunique()} PERMNOs')
print(f'Date range: {crsp["date"].min().date()} – {crsp["date"].max().date()}')

Pulling CRSP for 890 PERMNOs
Date range: 2002-12-15 – 2026-01-20


CRSP chunks:   0%|          | 0/2 [00:00<?, ?it/s]

CRSP monthly: 167,910 rows  |  849 PERMNOs
Date range: 2002-12-31 – 2024-12-31


## 6. Pull Compustat accounting data

Total assets (`at`), net income (`ni`), sales (`sale`), R&D (`xrd`), EBIT (`ebit`)
for the fiscal year ending before the filing date.

In [11]:
# Map PERMNO → GVKEY via CRSP-Compustat link
ccm = db.raw_sql("""
    SELECT gvkey, lpermno AS permno, linkdt, linkenddt, linktype, linkprim
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LU', 'LC')
      AND linkprim IN ('P', 'C')
""")
ccm['permno']    = ccm['permno'].astype(int)
ccm['linkdt']    = pd.to_datetime(ccm['linkdt'])
ccm['linkenddt'] = pd.to_datetime(ccm['linkenddt'], errors='coerce').fillna(
    pd.Timestamp('2099-12-31'))

# Get unique GVKEYs for our PERMNOs
gvkeys = ccm[ccm['permno'].isin(permnos)]['gvkey'].unique().tolist()
print(f'GVKEYs mapped: {len(gvkeys)}')

# Pull Compustat annual
gvkey_str = "','".join(gvkeys)
comp = db.raw_sql(f"""
    SELECT gvkey, datadate, fyear, at, ni, sale, xrd, ebit,
           ceq, txditc, pstkrv, pstkl, pstk
    FROM comp.funda
    WHERE gvkey IN ('{gvkey_str}')
      AND datafmt = 'STD'
      AND indfmt  = 'INDL'
      AND consol  = 'C'
      AND popsrc  = 'D'
      AND fyear BETWEEN {CONFIG['start_year']-1} AND {CONFIG['end_year']}
""")
comp['datadate'] = pd.to_datetime(comp['datadate'])

# Book equity (Fama-French definition)
comp['ps'] = comp['pstkrv'].fillna(comp['pstkl']).fillna(comp['pstk']).fillna(0)
comp['book_equity'] = comp['ceq'].fillna(0) + comp['txditc'].fillna(0) - comp['ps']
comp.loc[comp['book_equity'] <= 0, 'book_equity'] = np.nan

# Link GVKEY → PERMNO (time-aware)
comp = comp.merge(ccm[['gvkey', 'permno', 'linkdt', 'linkenddt']],
                  on='gvkey', how='inner')
comp = comp[(comp['datadate'] >= comp['linkdt']) &
            (comp['datadate'] <= comp['linkenddt'])]
comp = comp.drop(columns=['linkdt', 'linkenddt'])
comp = comp.sort_values('datadate').drop_duplicates(
    subset=['permno', 'fyear'], keep='last')

print(f'Compustat: {len(comp):,} firm-years  |  {comp["permno"].nunique()} PERMNOs')

GVKEYs mapped: 911
Compustat: 15,735 firm-years  |  948 PERMNOs


## 7. Build the final panel

For each filing:
- Attach the most recent Compustat data (fiscal year ≤ filing year)
- Pull the 12 monthly CRSP returns starting the **month after** the filing date
- Compute backward-looking momentum and reversal from CRSP

In [12]:
# ── Attach Compustat controls to filings ──────────────────────────
# Use most recent fiscal year ending on or before the filing date
comp_cols = ['permno', 'datadate', 'fyear', 'at', 'ni', 'sale',
             'xrd', 'ebit', 'book_equity']
comp_clean = comp[comp_cols].copy()

filing_comp = []
for _, row in tqdm(filings.iterrows(), total=len(filings), desc='Merging Compustat'):
    p = row['permno']
    fd = row['filing_date']
    # Most recent Compustat before filing date
    mask = (comp_clean['permno'] == p) & (comp_clean['datadate'] <= fd)
    match = comp_clean[mask].sort_values('datadate').tail(1)
    if len(match) == 1:
        m = match.iloc[0]
        filing_comp.append({
            'cik': row['cik'], 'year': row['year'], 'permno': p,
            'filing_date': fd,
            'at': m['at'], 'ni': m['ni'], 'sale': m['sale'],
            'xrd': m['xrd'], 'ebit': m['ebit'], 'book_equity': m['book_equity'],
        })
    else:
        filing_comp.append({
            'cik': row['cik'], 'year': row['year'], 'permno': p,
            'filing_date': fd,
            'at': np.nan, 'ni': np.nan, 'sale': np.nan,
            'xrd': np.nan, 'ebit': np.nan, 'book_equity': np.nan,
        })

filings_full = pd.DataFrame(filing_comp)
print(f'Filings with Compustat: {filings_full["at"].notna().sum():,} / {len(filings_full):,}')

# ── Build forward-return panel ────────────────────────────────────
# For each filing, get CRSP returns for months (fdate, fdate+12m]
crsp['month_end'] = crsp['date'] + pd.offsets.MonthEnd(0)
crsp_monthly = crsp[['permno', 'month_end', 'ret', 'mktcap']].copy()
crsp_monthly = crsp_monthly.sort_values(['permno', 'month_end'])

# Compute momentum (t-12 to t-2) and reversal (t-1) from CRSP
crsp_monthly['log_r'] = np.log1p(crsp_monthly['ret'].fillna(0))
g = crsp_monthly.groupby('permno')
crsp_monthly['reversal'] = g['ret'].shift(1)
crsp_monthly['momentum'] = np.expm1(
    g['log_r'].transform(lambda x: x.shift(2).rolling(11, min_periods=8).sum())
)

print('Building forward-return panel...')
panel_rows = []

for _, filing in tqdm(filings_full.iterrows(), total=len(filings_full),
                       desc='Aligning returns'):
    p  = filing['permno']
    fd = filing['filing_date']
    # Month after filing date through 12 months later
    start_month = fd + pd.offsets.MonthEnd(0) + pd.DateOffset(days=1)
    end_month   = fd + pd.DateOffset(months=12) + pd.offsets.MonthEnd(0)

    mask = ((crsp_monthly['permno'] == p) &
            (crsp_monthly['month_end'] > fd) &
            (crsp_monthly['month_end'] <= end_month))
    firm_rets = crsp_monthly[mask].copy()

    for _, mr in firm_rets.iterrows():
        panel_rows.append({
            'cik':            filing['cik'],
            'year':           filing['year'],
            'permno':         p,
            'filing_date':    fd,
            'return_month':   mr['month_end'],
            'monthly_return': mr['ret'],
            'mktcap':         mr['mktcap'],
            'momentum':       mr['momentum'],
            'reversal':       mr['reversal'],
            # Compustat controls (same for all 12 months of this filing)
            'at':             filing['at'],
            'ni':             filing['ni'],
            'sale':           filing['sale'],
            'xrd':            filing['xrd'],
            'ebit':           filing['ebit'],
            'book_equity':    filing['book_equity'],
        })

panel = pd.DataFrame(panel_rows)
print(f'\nFinal panel: {len(panel):,} firm-month obs')
print(f'  CIKs:   {panel["cik"].nunique():,}')
print(f'  Months: {panel["return_month"].nunique():,}')
print(f'  Period: {panel["return_month"].min().date()} – {panel["return_month"].max().date()}')

Merging Compustat:   0%|          | 0/10264 [00:00<?, ?it/s]

Filings with Compustat: 9,591 / 10,264
Building forward-return panel...


Aligning returns:   0%|          | 0/10264 [00:00<?, ?it/s]


Final panel: 119,252 firm-month obs
  CIKs:   839
  Months: 252
  Period: 2004-01-31 – 2024-12-31


## 8. Derived controls

In [14]:
panel['mktcap'] = pd.to_numeric(panel['mktcap'], errors='coerce')
panel['at'] = pd.to_numeric(panel['at'], errors='coerce')
panel['ni'] = pd.to_numeric(panel['ni'], errors='coerce')
panel['book_equity'] = pd.to_numeric(panel['book_equity'], errors='coerce')

# Log market cap
panel['log_mktcap'] = np.nan
mask = panel['mktcap'] > 0
panel.loc[mask, 'log_mktcap'] = np.log(panel.loc[mask, 'mktcap'])

# Profitability (ROA)
panel['profitability'] = np.nan
mask = panel['at'] > 0
panel.loc[mask, 'profitability'] = panel.loc[mask, 'ni'] / panel.loc[mask, 'at']

# Book-to-market
panel['bm'] = np.nan
mask = (panel['mktcap'] > 0) & panel['book_equity'].notna()
panel.loc[mask, 'bm'] = panel.loc[mask, 'book_equity'] / panel.loc[mask, 'mktcap']

# Log assets
panel['log_at'] = np.nan
mask = panel['at'] > 0
panel.loc[mask, 'log_at'] = np.log(panel.loc[mask, 'at'])

print('Derived controls computed.')
print(panel[['monthly_return', 'log_mktcap', 'bm', 'profitability',
             'momentum', 'reversal']].describe().round(4))

Derived controls computed.
        log_mktcap           bm  profitability     momentum
count  119047.0000  115328.0000    118646.0000  118997.0000
mean        9.7838       0.4821         0.0637       0.1242
std         1.1794       0.6137         0.0764       0.3451
min         3.7776       0.0000        -0.8526      -0.9896
25%         9.0151       0.2000         0.0238      -0.0627
50%         9.6729       0.3596         0.0558       0.1049
75%        10.4646       0.6197         0.0992       0.2771
max        15.1466      84.7966         0.5111      10.4380


## 9. Save

In [15]:
panel.to_parquet(CONFIG['output_path'], index=False, compression='snappy')
print(f'Saved: {CONFIG["output_path"]}')
print(f'  {len(panel):,} rows  |  {panel["cik"].nunique():,} CIKs  |  '
      f'{panel["return_month"].nunique():,} months')

# Sanity check: firms per month
monthly_counts = panel.groupby('return_month')['cik'].nunique()
print(f'\nFirms per month:')
print(f'  Mean:   {monthly_counts.mean():.0f}')
print(f'  Median: {monthly_counts.median():.0f}')
print(f'  Min:    {monthly_counts.min()}')
print(f'  Months with >= 30 firms:  {(monthly_counts >= 30).sum()}')
print(f'  Months with >= 100 firms: {(monthly_counts >= 100).sum()}')
print(f'  Months with >= 200 firms: {(monthly_counts >= 200).sum()}')

db.close()
print('\nWRDS connection closed. Done.')

Saved: /content/drive/MyDrive/FML_project_4/crsp_returns_panel.parquet
  119,252 rows  |  839 CIKs  |  252 months

Firms per month:
  Mean:   440
  Median: 448
  Min:    5
  Months with >= 30 firms:  251
  Months with >= 100 firms: 250
  Months with >= 200 firms: 250

WRDS connection closed. Done.
